In [18]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor

In [19]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print(train.shape)
print(test.shape)

(1460, 81)
(1459, 80)


In [20]:
test_ids = test["Id"]

In [21]:
y = np.log1p(train["SalePrice"])

In [22]:
train = train.drop("SalePrice", axis=1)

In [23]:
def preprocess(df):
    df = df.copy()

    # Drop columns removed during EDA
    df = df.drop(
        columns=['Id', 'PoolQC', 'MiscFeature', 'Alley', 'Fence'],
        errors='ignore'
    )

    # Missing value handling

    # MasVnrType
    df['MasVnrType'] = df['MasVnrType'].fillna('None')

    # LotFrontage
    df['LotFrontage'] = df['LotFrontage'].fillna(
        df['LotFrontage'].median()
    )

    # FireplaceQu
    df['FireplaceQu'] = df['FireplaceQu'].fillna('None')

    # Garage categorical columns
    garage_cols = [
        'GarageType',
        'GarageFinish',
        'GarageQual',
        'GarageCond'
    ]

    for col in garage_cols:
        df[col] = df[col].fillna('None')

    # Garage numerical column
    df['GarageYrBlt'] = df['GarageYrBlt'].fillna(
        df['GarageYrBlt'].median()
    )

    # Basement categorical columns
    bsmt_cols = [
        'BsmtQual',
        'BsmtCond',
        'BsmtExposure',
        'BsmtFinType1',
        'BsmtFinType2'
    ]

    for col in bsmt_cols:
        df[col] = df[col].fillna('None')

    # MasVnrArea
    df['MasVnrArea'] = df['MasVnrArea'].fillna(
        df['MasVnrArea'].median()
    )

    # Electrical
    df['Electrical'] = df['Electrical'].fillna(
        df['Electrical'].mode()[0]
    )

    # Feature Engineering

    df['TotalSF'] = (
        df['TotalBsmtSF']
        + df['1stFlrSF']
        + df['2ndFlrSF']
    )

    df['HouseAge'] = (
        df['YrSold']
        - df['YearBuilt']
    )

    df['YearsSinceRemodel'] = (
        df['YrSold']
        - df['YearRemodAdd']
    )

    df['TotalBathrooms'] = (
        df['FullBath']
        + 0.5 * df['HalfBath']
        + df['BsmtFullBath']
        + 0.5 * df['BsmtHalfBath']
    )

    df['TotalPorchSF'] = (
        df['OpenPorchSF']
        + df['EnclosedPorch']
        + df['3SsnPorch']
        + df['ScreenPorch']
    )

    return df

In [24]:
train = preprocess(train)
test = preprocess(test)

In [25]:
combined = pd.concat(
    [train, test],
    axis=0,
    ignore_index=True
)

print(combined.shape)

(2919, 80)


In [26]:
combined.isnull().sum().sort_values(
    ascending=False
).head(20)

MSZoning             4
BsmtFullBath         2
TotalBathrooms       2
BsmtHalfBath         2
Utilities            2
Functional           2
Exterior1st          1
TotalBsmtSF          1
BsmtUnfSF            1
BsmtFinSF2           1
KitchenQual          1
BsmtFinSF1           1
GarageCars           1
GarageArea           1
Exterior2nd          1
TotalSF              1
SaleType             1
EnclosedPorch        0
MiscVal              0
YearsSinceRemodel    0
dtype: int64

In [27]:
# MSZoning
combined['MSZoning'] = combined['MSZoning'].fillna(
    combined['MSZoning'].mode()[0]
)

# Utilities
combined['Utilities'] = combined['Utilities'].fillna(
    combined['Utilities'].mode()[0]
)

# Functional
combined['Functional'] = combined['Functional'].fillna(
    combined['Functional'].mode()[0]
)

# Exterior
combined['Exterior1st'] = combined['Exterior1st'].fillna(
    combined['Exterior1st'].mode()[0]
)

combined['Exterior2nd'] = combined['Exterior2nd'].fillna(
    combined['Exterior2nd'].mode()[0]
)

# KitchenQual
combined['KitchenQual'] = combined['KitchenQual'].fillna(
    combined['KitchenQual'].mode()[0]
)

# SaleType
combined['SaleType'] = combined['SaleType'].fillna(
    combined['SaleType'].mode()[0]
)

# Basement numerical
for col in [
    'BsmtFinSF1',
    'BsmtFinSF2',
    'BsmtUnfSF',
    'TotalBsmtSF',
    'BsmtFullBath',
    'BsmtHalfBath'
]:
    combined[col] = combined[col].fillna(
        combined[col].median()
    )

# Garage numerical
for col in [
    'GarageCars',
    'GarageArea'
]:
    combined[col] = combined[col].fillna(
        combined[col].median()
    )

In [28]:
combined = pd.get_dummies(
    combined,
    drop_first=True
)

print(combined.shape)

(2919, 251)


In [29]:
X_train = combined.iloc[:len(train)]
X_test = combined.iloc[len(train):]

print(X_train.shape)
print(X_test.shape)

(1460, 251)
(1459, 251)


In [30]:
from xgboost import XGBRegressor

best_xgb = XGBRegressor(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.01,
    subsample=0.7,
    colsample_bytree=0.7,
    random_state=42,
    objective='reg:squarederror'
)

best_xgb.fit(X_train, y)

,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.7
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None
,feature_types,None


In [31]:
preds = best_xgb.predict(X_test)
preds = np.expm1(preds)

In [32]:
submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": preds
})

submission.head()


,Id,SalePrice
0,1461,123397.601562
1,1462,160240.140625
2,1463,182484.953125
3,1464,193296.484375
4,1465,185137.515625


In [33]:
submission.to_csv(
    "submission.csv",
    index=False
)

In [34]:
import joblib

joblib.dump(best_xgb, "xgb_house_price.pkl")

['xgb_house_price.pkl']

In [35]:
joblib.dump(X_train.columns.tolist(), "feature_columns.pkl")

['feature_columns.pkl']

## DEPLOYMENT MODEL

In [37]:
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': best_xgb.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

feature_importance.head(20)

,Feature,Importance
3,OverallQual,0.140252
36,TotalSF,0.092380
155,ExterQual_TA,0.079123
198,CentralAir_Y,0.038907
25,GarageCars,0.036645
205,KitchenQual_TA,0.035189
39,TotalBathrooms,0.022997
15,GrLivArea,0.019216
223,GarageFinish_None,0.018542
214,FireplaceQu_None,0.017092


In [38]:
deploy_features = [
    'OverallQual',
    'TotalSF',
    'GarageCars',
    'TotalBathrooms',
    'GrLivArea',
    'HouseAge',
    'Fireplaces'
]

X_deploy = X_train[deploy_features]

In [39]:
from xgboost import XGBRegressor

deploy_model = XGBRegressor(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    random_state=42,
    objective='reg:squarederror'
)

deploy_model.fit(X_deploy, y)

,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None
,feature_types,None


In [42]:
joblib.dump(deploy_model, "deploy_model.pkl")

['deploy_model.pkl']

In [43]:
pred = deploy_model.predict(
    pd.DataFrame({
        'OverallQual':[7],
        'TotalSF':[2500],
        'GarageCars':[2],
        'TotalBathrooms':[2.5],
        'GrLivArea':[1800],
        'HouseAge':[15],
        'Fireplaces':[1]
    })
)

print(np.expm1(pred))

[192286.72]
